# Data preparation

Owner: Oscar. Reads the **bronze** layer and writes the cleaned **silver** layer used by the other notebooks.

In [1]:
import sys
sys.path.insert(0, "../jobs")  # notebooks/ -> jobs/

from pathlib import Path

from pyspark.sql import DataFrame, functions as F
from pyspark.sql.types import ArrayType, StringType

from common import DATASETS, SILVER_DIR, PROJECT_ROOT, get_spark, read_bronze

# 2 cores and 2 GB: slower but the JVM is no longer killed by Docker (out of RAM)
spark = get_spark("imdb_data_preparation", driver_memory="2g", cores=2)
spark.sparkContext.setLogLevel("WARN")
# more, smaller shuffle tasks: principals has ~100 M rows and 16 partitions are too big for 4 GB
spark.conf.set("spark.sql.shuffle.partitions", "200")

# small result tables (csv) reused in REPORT.md
RESULTS_DIR = PROJECT_ROOT / "results" / "preparation"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

bronze = {name: read_bronze(spark, name) for name in DATASETS}
for name, df in bronze.items():
    print(f"{name:18s} {df.count():>12,} rows")

title.basics         12,951,113 rows
title.akas           60,321,594 rows
title.crew           12,949,786 rows
title.episode        10,025,088 rows
title.principals    102,649,609 rows
title.ratings         1,730,885 rows
name.basics          15,802,420 rows


## Parameters

Every threshold used below is defined here once, so the choices are visible and easy to change.

In [2]:
CURRENT_YEAR = 2026
FEATURE_MIN_RUNTIME = 40      # feature-length film: runtime > 40 min (Academy / AFI / BFI)
MAX_PLAUSIBLE_RUNTIME = 600   # runtimes above 10 h are flagged as suspicious
MAX_LIFESPAN = 120            # years between birth and death
FIRST_FILM_YEAR = 1874        # earliest year a motion picture can be dated

## 1. For each column of each table in the IMDB dataset, calculate the percentage of nulls, missing, or empty records.

For each column, a value is counted as missing when it is:

- `null` (IMDb writes missing values as `\N`, read as null in the bronze layer);
- an empty or whitespace-only string;
- an empty array (list columns such as `genres`).

In [3]:
def missing_profile(name: str, df: DataFrame) -> DataFrame:
    """One row per column: number and percentage of missing values."""
    total = df.count()
    exprs = []
    for field in df.schema.fields:
        c = F.col(field.name)
        if isinstance(field.dataType, StringType):
            is_missing = c.isNull() | (F.trim(c) == "")
        elif isinstance(field.dataType, ArrayType):
            is_missing = c.isNull() | (F.size(c) == 0)
        else:
            is_missing = c.isNull()
        exprs.append(F.sum(is_missing.cast("long")).alias(field.name))
    counts = df.agg(*exprs).first().asDict()  # one pass over the table
    rows = [(name, column, total, int(n), 100.0 * n / total if total else 0.0)
            for column, n in counts.items()]
    return spark.createDataFrame(
        rows, "table string, column string, rows long, missing long, pct_missing double")


profiles = None
for name, df in bronze.items():
    p = missing_profile(name, df)
    profiles = p if profiles is None else profiles.unionByName(p)

profiles = profiles.withColumn("pct_missing", F.round("pct_missing", 2)).cache()
profiles.orderBy("table", F.desc("pct_missing")).show(100, truncate=False)
profiles.toPandas().to_csv(RESULTS_DIR / "missing_per_column.csv", index=False)

+----------------+-----------------+---------+--------+-----------+
|table           |column           |rows     |missing |pct_missing|
+----------------+-----------------+---------+--------+-----------+
|name.basics     |deathYear        |15802420 |15537057|98.32      |
|name.basics     |birthYear        |15802420 |15116831|95.66      |
|name.basics     |primaryProfession|15802420 |3216642 |20.36      |
|name.basics     |knownForTitles   |15802420 |1915685 |12.12      |
|name.basics     |nconst           |15802420 |0       |0.0        |
|name.basics     |primaryName      |15802420 |89      |0.0        |
|title.akas      |attributes       |60321594 |60004431|99.47      |
|title.akas      |types            |60321594 |40342236|66.88      |
|title.akas      |language         |60321594 |19639034|32.56      |
|title.akas      |region           |60321594 |12991240|21.54      |
|title.akas      |ordering         |60321594 |0       |0.0        |
|title.akas      |title            |60321594 |0 

## 2. Which five columns across the entire dataset have the highest percentage of null values?

In [4]:
top5 = profiles.orderBy(F.desc("pct_missing")).limit(5)
top5.show(truncate=False)
top5.toPandas().to_csv(RESULTS_DIR / "top5_missing_columns.csv", index=False)

+----------------+----------+---------+--------+-----------+
|table           |column    |rows     |missing |pct_missing|
+----------------+----------+---------+--------+-----------+
|title.akas      |attributes|60321594 |60004431|99.47      |
|title.basics    |endYear   |12951113 |12786793|98.73      |
|name.basics     |deathYear |15802420 |15537057|98.32      |
|name.basics     |birthYear |15802420 |15116831|95.66      |
|title.principals|job       |102649609|82794746|80.66      |
+----------------+----------+---------+--------+-----------+



## 3. Identify other data quality issues for the dataset such as principals who died before they were born, foreign keys with missing parent records, etc.

Each check below returns the rows concerned; the counts are gathered in one summary table at the end of the section.

In [5]:
titles = bronze["title.basics"]
names = bronze["name.basics"]
principals = bronze["title.principals"]
ratings = bronze["title.ratings"]
episodes = bronze["title.episode"]
crew = bronze["title.crew"]
akas = bronze["title.akas"]

issues = []  # (table, check, rows)


def record(table, check, df):
    n = df.count()
    issues.append((table, check, n))
    print(f"{table:18s} {check:60s} {n:>12,}")
    return df

### 3.1 People: impossible birth and death years

In [6]:
died_before_born = record("name.basics", "deathYear < birthYear", names.filter(F.col("deathYear") < F.col("birthYear")))
died_before_born.select("nconst", "primaryName", "birthYear", "deathYear", "primaryProfession").show(20, truncate=False)

record("name.basics", f"lifespan > {MAX_LIFESPAN} years",
       names.filter(F.col("deathYear") - F.col("birthYear") > MAX_LIFESPAN))
record("name.basics", f"birthYear > {CURRENT_YEAR}", names.filter(F.col("birthYear") > CURRENT_YEAR))
record("name.basics", f"deathYear > {CURRENT_YEAR}", names.filter(F.col("deathYear") > CURRENT_YEAR))
alive_too_old = record("name.basics", f"no deathYear and born more than {MAX_LIFESPAN} years ago",
                       names.filter(F.col("deathYear").isNull() & (F.col("birthYear") < CURRENT_YEAR - MAX_LIFESPAN)))

name.basics        deathYear < birthYear                                                  28
+----------+---------------------+---------+---------+--------------------------+
|nconst    |primaryName          |birthYear|deathYear|primaryProfession         |
+----------+---------------------+---------+---------+--------------------------+
|nm0012635 |Aesop                |620      |560      |[writer]                  |
|nm18335674|Aleksander Normak    |1995     |1984     |[actor]                   |
|nm1419271 |Socrates             |470      |399      |[writer]                  |
|nm1414200 |Xenophon             |430      |354      |[writer]                  |
|nm14888413|Smith E. Johnson     |1953     |1917     |[archive_footage]         |
|nm0814668 |Sophocles            |496      |406      |[writer, music_department]|
|nm0515385 |Titus Livius         |59       |17       |[writer]                  |
|nm17520180|B.N. MacLarty        |1899     |1873     |NULL                      |
|nm00

### 3.2 Principals credited on titles released before their birth, or long after their death

In [7]:
credits = (
    principals.select("tconst", "nconst", "category")
    .join(titles.select("tconst", "startYear", "titleType"), "tconst")
    .join(names.select("nconst", "primaryName", "birthYear", "deathYear"), "nconst")
)

before_birth = record("title.principals", "credited on a title released before their birth",
                      credits.filter(F.col("startYear") < F.col("birthYear")))
before_birth.orderBy(F.col("birthYear") - F.col("startYear"), ascending=False).show(10, truncate=False)

# archive footage / archive sound are legitimately posthumous
after_death = record("title.principals", "credited > 2 years after death (excluding archive_*)",
                     credits.filter((F.col("startYear") > F.col("deathYear") + 2)
                                    & ~F.col("category").startswith("archive")))
after_death.groupBy("category").count().orderBy(F.desc("count")).show()

title.principals   credited on a title released before their birth                     3,101
+----------+----------+--------+---------+---------+-------------------------+---------+---------+
|nconst    |tconst    |category|startYear|titleType|primaryName              |birthYear|deathYear|
+----------+----------+--------+---------+---------+-------------------------+---------+---------+
|nm3151514 |tt0467064 |actor   |1914     |short    |Younus Nabi              |1998     |NULL     |
|nm10990240|tt38343462|self    |1927     |tvSeries |Jayden Daniels           |2000     |NULL     |
|nm1808213 |tt0016392 |composer|1924     |movie    |Michael-Alexander Münster|1992     |NULL     |
|nm0242155 |tt0000417 |composer|1902     |short    |Jean-Benoît Dunckel      |1969     |NULL     |
|nm0324073 |tt0000417 |composer|1902     |short    |Nicolas Godin            |1969     |NULL     |
|nm1311785 |tt0002208 |composer|1912     |short    |Lev Zhurbin              |1978     |NULL     |
|nm1976494 |tt02

### 3.3 Titles: impossible years and runtimes

In [8]:
record("title.basics", f"startYear > {CURRENT_YEAR} (announced, not released)", titles.filter(F.col("startYear") > CURRENT_YEAR))
record("title.basics", f"startYear < {FIRST_FILM_YEAR}", titles.filter(F.col("startYear") < FIRST_FILM_YEAR))
record("title.basics", "endYear < startYear", titles.filter(F.col("endYear") < F.col("startYear")))
record("title.basics", "endYear set on a non-series title",
       titles.filter(F.col("endYear").isNotNull() & ~F.col("titleType").isin("tvSeries", "tvMiniSeries")))
record("title.basics", "runtimeMinutes <= 0", titles.filter(F.col("runtimeMinutes") <= 0))
long_runtime = record("title.basics", f"runtimeMinutes > {MAX_PLAUSIBLE_RUNTIME}",
                      titles.filter(F.col("runtimeMinutes") > MAX_PLAUSIBLE_RUNTIME))
long_runtime.groupBy("titleType").count().orderBy(F.desc("count")).show()
long_runtime.filter(F.col("titleType") == "movie").orderBy(F.desc("runtimeMinutes")) \
    .select("tconst", "primaryTitle", "startYear", "runtimeMinutes").show(10, truncate=False)
record("title.basics", "isAdult not in (0, 1)", titles.filter(~F.col("isAdult").isin(0, 1)))
record("title.basics", "more than 3 genres", titles.filter(F.size("genres") > 3))

title.basics       startYear > 2026 (announced, not released)                          6,017
title.basics       startYear < 1874                                                        0
title.basics       endYear < startYear                                                     0
title.basics       endYear set on a non-series title                                       0
title.basics       runtimeMinutes <= 0                                                     4
title.basics       runtimeMinutes > 600                                                1,133
+------------+-----+
|   titleType|count|
+------------+-----+
|       video|  370|
|    tvSeries|  259|
|   tvEpisode|  165|
|       movie|  109|
|   tvSpecial|  107|
|tvMiniSeries|   90|
|     tvMovie|   29|
|   videoGame|    4|
+------------+-----+

+----------+-------------------------------+---------+--------------+
|tconst    |primaryTitle                   |startYear|runtimeMinutes|
+----------+-------------------------------+-----

DataFrame[tconst: string, titleType: string, primaryTitle: string, originalTitle: string, isAdult: int, startYear: int, endYear: int, runtimeMinutes: int, genres: array<string>]

### 3.4 Ratings and episodes

In [9]:
record("title.ratings", "averageRating outside [1, 10]", ratings.filter(~F.col("averageRating").between(1, 10)))
record("title.ratings", "numVotes <= 0", ratings.filter(F.col("numVotes") <= 0))
record("title.episode", "seasonNumber or episodeNumber missing",
       episodes.filter(F.col("seasonNumber").isNull() | F.col("episodeNumber").isNull()))
record("title.episode", "parent is not a series",
       episodes.join(titles.select(F.col("tconst").alias("parentTconst"), "titleType"), "parentTconst")
       .filter(~F.col("titleType").isin("tvSeries", "tvMiniSeries")))

title.ratings      averageRating outside [1, 10]                                           0
title.ratings      numVotes <= 0                                                           0
title.episode      seasonNumber or episodeNumber missing                           2,120,177
title.episode      parent is not a series                                                  0


DataFrame[parentTconst: string, tconst: string, seasonNumber: int, episodeNumber: int, titleType: string]

### 3.5 Duplicated primary keys

In [10]:
def duplicates(df, keys):
    return df.groupBy(*keys).count().filter("count > 1")

record("title.basics", "duplicated tconst", duplicates(titles, ["tconst"]))
record("name.basics", "duplicated nconst", duplicates(names, ["nconst"]))
record("title.ratings", "duplicated tconst", duplicates(ratings, ["tconst"]))
record("title.episode", "duplicated tconst", duplicates(episodes, ["tconst"]))
record("title.crew", "duplicated tconst", duplicates(crew, ["tconst"]))
record("title.principals", "duplicated (tconst, ordering)", duplicates(principals, ["tconst", "ordering"]))
record("title.akas", "duplicated (titleId, ordering)", duplicates(akas, ["titleId", "ordering"]))

title.basics       duplicated tconst                                                       0
name.basics        duplicated nconst                                                       0
title.ratings      duplicated tconst                                                       0
title.episode      duplicated tconst                                                       0
title.crew         duplicated tconst                                                       0
title.principals   duplicated (tconst, ordering)                                           0
title.akas         duplicated (titleId, ordering)                                          0


DataFrame[titleId: string, ordering: int, count: bigint]

### 3.6 Foreign keys with missing parent records

A `left_anti` join keeps the child rows that have **no** matching parent. List columns (directors, writers, knownForTitles) are exploded first.

In [11]:
title_ids = titles.select(F.col("tconst").alias("key")).distinct()
name_ids = names.select(F.col("nconst").alias("key")).distinct()


def orphans(child, column, parent_ids, explode=False):
    keys = child.select(F.explode(column).alias("key")) if explode else child.select(F.col(column).alias("key"))
    return keys.filter(F.col("key").isNotNull()).join(parent_ids, "key", "left_anti")


FOREIGN_KEYS = [
    ("title.ratings", ratings, "tconst", title_ids, False),
    ("title.akas", akas, "titleId", title_ids, False),
    ("title.crew", crew, "tconst", title_ids, False),
    ("title.crew", crew, "directors", name_ids, True),
    ("title.crew", crew, "writers", name_ids, True),
    ("title.episode", episodes, "tconst", title_ids, False),
    ("title.episode", episodes, "parentTconst", title_ids, False),
    ("title.principals", principals, "tconst", title_ids, False),
    ("title.principals", principals, "nconst", name_ids, False),
    ("name.basics", names, "knownForTitles", title_ids, True),
]
for table, df, column, parent, explode in FOREIGN_KEYS:
    target = "title.basics" if parent is title_ids else "name.basics"
    record(table, f"{column} -> {target}: missing parent", orphans(df, column, parent, explode))

title.ratings      tconst -> title.basics: missing parent                                  0
title.akas         titleId -> title.basics: missing parent                                 2
title.crew         tconst -> title.basics: missing parent                                 30
title.crew         directors -> name.basics: missing parent                              423
title.crew         writers -> name.basics: missing parent                              1,149
title.episode      tconst -> title.basics: missing parent                                  0
title.episode      parentTconst -> title.basics: missing parent                            5
title.principals   tconst -> title.basics: missing parent                                  0
title.principals   nconst -> name.basics: missing parent                               8,441
name.basics        knownForTitles -> title.basics: missing parent                         69


### Summary of the data quality checks

In [12]:
summary = spark.createDataFrame(issues, "table string, check string, rows long")
summary.orderBy("table", F.desc("rows")).show(100, truncate=False)
summary.toPandas().to_csv(RESULTS_DIR / "quality_checks.csv", index=False)

+----------------+----------------------------------------------------+-------+
|table           |check                                               |rows   |
+----------------+----------------------------------------------------+-------+
|name.basics     |no deathYear and born more than 120 years ago       |2793   |
|name.basics     |knownForTitles -> title.basics: missing parent      |69     |
|name.basics     |deathYear < birthYear                               |28     |
|name.basics     |lifespan > 120 years                                |2      |
|name.basics     |duplicated nconst                                   |0      |
|name.basics     |birthYear > 2026                                    |0      |
|name.basics     |deathYear > 2026                                    |0      |
|title.akas      |titleId -> title.basics: missing parent             |2      |
|title.akas      |duplicated (titleId, ordering)                      |0      |
|title.basics    |startYear > 2026 (anno

## 4. Design and implement a way to handle these values.

Rules applied to build the silver layer (the reasons are given in REPORT.md):

| Issue | Handling |
|---|---|
| empty strings, empty arrays | turned into `null`, so every missing value has one representation |
| duplicated primary keys | one row kept per key |
| child rows whose foreign key has no parent | removed (they cannot be joined); list columns keep only the ids that exist |
| impossible birth / death years | kept as they are, flagged with `has_invalid_life_dates` |
| `runtimeMinutes <= 0` | set to `null` |
| runtime above `MAX_PLAUSIBLE_RUNTIME` | kept, flagged with `is_runtime_suspicious` |
| `startYear` after `CURRENT_YEAR` | kept, flagged with `is_unreleased` |
| `averageRating` outside [1, 10], `numVotes <= 0` | removed |
| missing ratings, runtimes, years | **not imputed**: left as `null` |

The silver `title.basics` also gets the shared definitions used by every section: `is_movie` and `is_feature_film` (movie with a runtime > `FEATURE_MIN_RUNTIME`).

In [3]:
def normalize_missing(df: DataFrame) -> DataFrame:
    """Empty / blank strings and empty arrays become null."""
    for field in df.schema.fields:
        c = F.col(field.name)
        if isinstance(field.dataType, StringType):
            df = df.withColumn(field.name, F.when(F.trim(c) == "", None).otherwise(c))
        elif isinstance(field.dataType, ArrayType):
            df = df.withColumn(field.name, F.when(F.size(c) == 0, None).otherwise(c))
    return df


clean = {name: normalize_missing(df) for name, df in bronze.items()}

# duplicated keys: dropDuplicates shuffles every column of the table, which is
# very heavy on title.principals (~100 M rows). We first look for duplicates on
# the key columns only, and only deduplicate when some exist.
def dedupe(df, keys):
    has_duplicates = df.groupBy(*keys).count().filter("count > 1").limit(1).count() > 0
    return df.dropDuplicates(keys) if has_duplicates else df


PRIMARY_KEYS = {
    "title.basics": ["tconst"],
    "name.basics": ["nconst"],
    "title.ratings": ["tconst"],
    "title.episode": ["tconst"],
    "title.crew": ["tconst"],
    "title.principals": ["tconst", "ordering"],
    "title.akas": ["titleId", "ordering"],
}
for name, keys in PRIMARY_KEYS.items():
    clean[name] = dedupe(clean[name], keys)

In [4]:
# title.basics: runtimes, flags and shared definitions
clean["title.basics"] = (
    clean["title.basics"]
    .withColumn("runtimeMinutes", F.when(F.col("runtimeMinutes") > 0, F.col("runtimeMinutes")))
    .withColumn("is_runtime_suspicious", F.coalesce(F.col("runtimeMinutes") > MAX_PLAUSIBLE_RUNTIME, F.lit(False)))
    .withColumn("is_unreleased", F.coalesce(F.col("startYear") > CURRENT_YEAR, F.lit(False)))
    .withColumn("is_movie", F.col("titleType") == "movie")
    .withColumn("is_feature_film", F.coalesce(
        (F.col("titleType") == "movie") & (F.col("runtimeMinutes") > FEATURE_MIN_RUNTIME), F.lit(False)))
)

# name.basics: flag impossible life dates instead of guessing the right value
clean["name.basics"] = clean["name.basics"].withColumn(
    "has_invalid_life_dates",
    F.coalesce(
        (F.col("deathYear") < F.col("birthYear"))
        | (F.col("deathYear") - F.col("birthYear") > MAX_LIFESPAN)
        | (F.col("birthYear") > CURRENT_YEAR)
        | (F.col("deathYear") > CURRENT_YEAR),
        F.lit(False)),
)

# title.ratings: impossible values
clean["title.ratings"] = clean["title.ratings"].filter(
    F.col("averageRating").between(1, 10) & (F.col("numVotes") > 0))

### Write the silver layer

In [5]:
# Write the silver layer table by table, in dependency order:
# the parent tables (title.basics, name.basics) are written first and the
# children are filtered against what was actually written, which keeps each
# Spark job small instead of one huge lineage.
OVERWRITE_SILVER = False  # True to rebuild tables that already exist in data/silver

spark.catalog.clearCache()  # free the memory used by the profiling step


def write_silver(name, df):
    target = SILVER_DIR / name
    if (target / "_SUCCESS").exists() and not OVERWRITE_SILVER:
        print(f"{name:18s} already in silver, skipped")
    else:
        df.write.mode("overwrite").parquet(str(target))
        print(f"{name:18s} written")
    return spark.read.parquet(str(target))


def keep_existing(df, column, parent, parent_key):
    """Drop the rows whose foreign key has no parent record."""
    # 1) find the (few) missing ids, working on the distinct keys only
    missing = (
        df.select(column).distinct()
        .join(parent.withColumnRenamed(parent_key, column), column, "left_anti")
    )
    # 2) remove them with a broadcast join: no shuffle of the big table
    return df.join(F.broadcast(missing), column, "left_anti")


def filter_id_list(df, key, column, parent, parent_key):
    """Keep only the ids of a list column that exist in the parent table (order kept)."""
    exploded = (
        df.select(key, F.posexplode(column).alias("pos", parent_key))
        .join(parent, parent_key, "left_semi")
        .groupBy(key)
        .agg(F.sort_array(F.collect_list(F.struct("pos", parent_key))).alias("items"))
        .select(key, F.col(f"items.{parent_key}").alias(column))
    )
    return df.drop(column).join(exploded, key, "left").select(df.columns)

In [6]:
silver = {}
silver["title.basics"] = write_silver("title.basics", clean["title.basics"])
valid_titles = silver["title.basics"].select("tconst")

silver["name.basics"] = write_silver(
    "name.basics", filter_id_list(clean["name.basics"], "nconst", "knownForTitles", valid_titles, "tconst"))
valid_names = silver["name.basics"].select("nconst")

title.basics       already in silver, skipped
name.basics        already in silver, skipped


In [7]:
silver["title.ratings"] = write_silver(
    "title.ratings", keep_existing(clean["title.ratings"], "tconst", valid_titles, "tconst"))
silver["title.akas"] = write_silver(
    "title.akas", keep_existing(clean["title.akas"], "titleId", valid_titles, "tconst"))
silver["title.episode"] = write_silver(
    "title.episode",
    keep_existing(keep_existing(clean["title.episode"], "tconst", valid_titles, "tconst"),
                  "parentTconst", valid_titles, "tconst"))

crew = keep_existing(clean["title.crew"], "tconst", valid_titles, "tconst")
crew = filter_id_list(crew, "tconst", "directors", valid_names, "nconst")
crew = filter_id_list(crew, "tconst", "writers", valid_names, "nconst")
silver["title.crew"] = write_silver("title.crew", crew)

title.ratings      already in silver, skipped
title.akas         already in silver, skipped
title.episode      already in silver, skipped
title.crew         already in silver, skipped


In [8]:
# the biggest table (~100 M rows): written last, alone
silver["title.principals"] = write_silver(
    "title.principals",
    keep_existing(keep_existing(clean["title.principals"], "tconst", valid_titles, "tconst"),
                  "nconst", valid_names, "nconst"))

title.principals   written


### Compare row counts between bronze and silver

In [ ]:
before_after = []
for name in DATASETS:
    n_before = bronze[name].count()
    n_after = silver[name].count()
    before_after.append((name, n_before, n_after, n_before - n_after))

counts = spark.createDataFrame(before_after, "table string, bronze_rows long, silver_rows long, removed long") \
    .withColumn("pct_removed", F.round(100 * F.col("removed") / F.col("bronze_rows"), 3))
counts.show(truncate=False)
counts.toPandas().to_csv(RESULTS_DIR / "bronze_vs_silver.csv", index=False)

In [ ]:
# quick check of the shared definitions in the silver layer
silver_titles = spark.read.parquet(str(SILVER_DIR / "title.basics"))
silver_titles.groupBy("is_movie", "is_feature_film").count().orderBy("is_movie", "is_feature_film").show()